Nice — below is a **fully commented, step-by-step implementation** of a simple LSTM cell and using it inside a Keras `RNN` layer. I’ll show:

1. math & intuition,
2. a correct `MyLSTMCell` implementation (subclassing `tf.keras.layers.AbstractRNNCell`),
3. using it with `tf.keras.layers.RNN` (sequence output and state output),
4. a tiny toy training example (predict sum of inputs across time) so you can run & test it.

You can copy–paste the whole thing and run it as-is (requires TensorFlow 2.x).

---

# Implementation + explanation




In [3]:
"""
FULL LSTM FROM SCRATCH IN KERAS
Compatible with:
  - TensorFlow 2.13–2.17+
  - Keras 3.x
No AbstractRNNCell required.
"""

import tensorflow as tf
import numpy as np

tf.random.set_seed(0)
np.random.seed(0)

# =============================================================
# 1. LSTM Cell Implemented From Scratch
# =============================================================

class MyLSTMCell(tf.keras.layers.Layer):
    """
    A minimal LSTM cell implemented manually using basic gate equations.
    Works with tf.keras.layers.RNN because:
      - It defines state_size
      - It defines output_size
      - call() returns (h_t, [h_t, c_t])
    """

    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)

        # REQUIRED by Keras RNN wrapper:
        self.state_size = [self.units, self.units]  # (hidden_state, cell_state)
        self.output_size = self.units               # output = hidden_state

    def build(self, input_shape):
        """
        Called once to create the trainable weights.
        """
        input_dim = int(input_shape[-1])

        # Weight matrix for input → gates
        self.W = self.add_weight(
            name="W",
            shape=(input_dim, 4 * self.units),
            initializer="glorot_uniform"
        )

        # Weight matrix for hidden state → gates
        self.U = self.add_weight(
            name="U",
            shape=(self.units, 4 * self.units),
            initializer="orthogonal"
        )

        # Bias vector for gates
        self.b = self.add_weight(
            name="b",
            shape=(4 * self.units,),
            initializer="zeros"
        )

        # ---- Add +1.0 to forget gate bias (unit forget bias) ----
        # Gates order: [i, f, o, g]
        f_start = self.units
        f_end = 2 * self.units
        new_b = tf.concat([
            self.b[:f_start],
            self.b[f_start:f_end] + tf.ones((self.units,)),
            self.b[f_end:]
        ], axis=0)
        self.b.assign(new_b)

        super().build(input_shape)

    def call(self, inputs, states):
        """
        LSTM update rule for SINGLE time step:
        inputs: (batch, input_dim)
        states: [h_{t-1}, c_{t-1}]
        returns: (h_t, [h_t, c_t])
        """
        h_tm1, c_tm1 = states  # previous hidden and cell states

        # Combined affine transformation
        # z = [i, f, o, g]
        z = (
            tf.matmul(inputs, self.W) +
            tf.matmul(h_tm1, self.U) +
            self.b
        )

        # Split into 4 gate vectors
        z_i, z_f, z_o, z_g = tf.split(z, 4, axis=1)

        # Gate activations
        i = tf.sigmoid(z_i)  # input gate
        f = tf.sigmoid(z_f)  # forget gate
        o = tf.sigmoid(z_o)  # output gate
        g = tf.tanh(z_g)     # candidate cell state

        # Cell state update
        c_t = f * c_tm1 + i * g

        # Hidden state update
        h_t = o * tf.tanh(c_t)

        return h_t, [h_t, c_t]

# =============================================================
# 2. Test the cell with random data (sanity check)
# =============================================================

print("=== Sanity Check ===")
batch = 2
timesteps = 4
features = 3
units = 5

x = tf.random.normal((batch, timesteps, features))

cell = MyLSTMCell(units)
rnn = tf.keras.layers.RNN(cell, return_sequences=True, return_state=True)

outputs, last_h, last_c = rnn(x)

print("outputs:", outputs.shape)  # (2,4,5)
print("last_h:", last_h.shape)    # (2,5)
print("last_c:", last_c.shape)    # (2,5)
print()

# =============================================================
# 3. Small training example: Sequence → Sum
# =============================================================

"""
Task:
  Input:  sequence of length T with random values in [0,1]
  Output: single number = sum of the sequence

This tests whether our LSTM actually learns a meaningful mapping.
"""

print("=== Training Example: Predict Sequence Sum ===")

T = 10        # sequence length
N = 1024      # number of samples

# Generate data
X = np.random.rand(N, T, 1).astype(np.float32)
y = np.sum(X, axis=1).astype(np.float32)  # shape (N,1)

# Train/validation split
split = int(0.8 * N)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

# =============================================================
# 4. Build model using our custom cell
# =============================================================

inputs = tf.keras.Input(shape=(T, 1))
h = tf.keras.layers.RNN(MyLSTMCell(16), return_sequences=False)(inputs)
output = tf.keras.layers.Dense(1)(h)
model = tf.keras.Model(inputs, output)

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

print(model.summary())

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=2
)

# =============================================================
# 5. Test the model
# =============================================================

print("=== Model Evaluation ===")
val_loss, val_mae = model.evaluate(X_val, y_val, verbose=0)
print("Validation Loss:", val_loss)
print("Validation MAE :", val_mae)

# Try a new example
x_test = np.random.rand(1, T, 1).astype(np.float32)
true_sum = np.sum(x_test)
pred = model.predict(x_test)

print("\nTrue sum:", float(true_sum))
print("Predicted:", float(pred))


=== Sanity Check ===
outputs: (2, 4, 5)
last_h: (2, 5)
last_c: (2, 5)

=== Training Example: Predict Sequence Sum ===


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_1 (RNN)                     │ (None, 16)             │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,169 (4.57 KB)

 Trainable params: 1,169 (4.57 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
26/26 - 3s - 122ms/step - loss: 24.0519 - mae: 4.8181 - val_loss: 22.3249 - val_mae: 4.6472
Epoch 2/10
26/26 - 0s - 9ms/step - loss: 18.7668 - mae: 4.2417 - val_loss: 14.5389 - val_mae: 3.7331
Epoch 3/10
26/26 - 0s - 9ms/step - loss: 9.4705 - mae: 2.9342 - val_loss: 5.3765 - val_mae: 2.1689
Epoch 4/10
26/26 - 0s - 10ms/step - loss: 3.3368 - mae: 1.6059 - val_loss: 1.9218 - val_mae: 1.1689
Epoch 5/10
26/26 - 0s - 9ms/step - loss: 1.2679 - mae: 0.9005 - val_loss: 0.8726 - val_mae: 0.7440
Epoch 6/10
26/26 - 0s - 9ms/step - loss: 0.8249 - mae: 0.7304 - val_loss: 0.7508 - val_mae: 0.6811
Epoch 7/10
26/26 - 0s - 9ms/step - loss: 0.7919 - mae: 0.7195 - val_loss: 0.7368 - val_mae: 0.6749
Epoch 8/10
26/26 - 0s - 10ms/step - loss: 0.7774 - mae: 0.7134 - val_loss: 0.7241 - val_mae: 0.6688
Epoch 9/10
26/26 - 0s - 9ms/step - loss: 0.7595 - mae: 0.7046 - val_loss: 0.7069 - val_mae: 0.6608
Epoch 10/10
26/26 - 0s - 9ms/step - loss: 0.7363 - mae: 0.6938 - val_loss: 0.6812 - val_mae: 0.6

/tmp/ipython-input-959466896.py:195: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("Predicted:", float(pred))


# Notes, tips & explanations

* **Why combine linear transforms?** Combining `x @ W` and `h @ U` into one `z` then splitting into 4 parts is efficient and the standard implementation pattern.
* **Gate ordering:** In the code I used the order `[i, f, o, g]` for the split. That's arbitrary but consistent. If you change it, change the bias forget-slice accordingly.
* **Forget-bias trick:** Setting the forget gate bias to a positive value (commonly +1) at initialization helps the network remember information in early training.
* **Initializers:** `glorot_uniform` for input weights and `orthogonal` for recurrent weights are common choices.
* **Why subclass `AbstractRNNCell`?** It gives `state_size` and `output_size` semantics so Keras `RNN` can manage initial states, masking, and stacking easily.
* **Vectorization:** This implementation is vectorized across the batch dimension. The time loop is handled by `tf.keras.layers.RNN` (C-optimized), so it’s reasonably efficient. If you wrote your own Python `for t in range(T)` over timesteps inside a layer you’d be slower in eager mode.
* **Extensions:**

  * Add **peephole connections** (use `c_{t-1}` in gate linear combination).
  * Add **layer normalization** inside gates.
  * Add **dropout** between time steps (use Keras `RNN` dropout/probability options or implement recurrent dropout).
  * Implement **coupled forget/input gates** (a variant).
* **Testing vs built-in:** For production you’d normally use `tf.keras.layers.LSTM` which is well-optimized and supports many options (CuDNN acceleration, masking, dropout, etc.). This custom implementation is educational and gives control for experimentation.